In [8]:
import json
import pandas as pd
from collections import defaultdict

# ====================== 你的配置 ======================
MODELS = [
    "qwen2.5-vl-72b",
    "qwen3-vl-32b",
    "qwen3.5-27b",
    "qwen3-vl-8b",
    "glm4.6v",
    "conan",
    "qwen3-vl-32b-thinking",
    "qwen3-vl-8b-thinking",
    "gemini-3.1-pro",
    "longvideoR1"
]

DRAMA_NAMES = [
    "kuang_biao", "huan_le_song", "zhan_chang_sha",
    "yi_qi_tong_guo_chuang_1", "ren_shi_jian", "friends",
    "da_qin_di_guo", "lost", "downton_abbey",
    "san_ti", "chen_mo_de_zhen_xiang", "shan_hai_qing", "zhen_huan_zhuan"
]

LEVELS = [3, 4, 5, 6]
RESOLUTIONS = ['256', '512', 'DA']
JSON_FILE = "nested_result_without_qa_details.json"  # 你的输入文件
EXCEL_FILE = "model_performance_report.xlsx"
# ======================================================

def load_json(path):
    """加载JSON文件"""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"❌ 加载JSON失败：{e}")
        return {}

def collect_all_drama_data(data):
    """收集每部剧的完整数据（包含total_stat和mini_batch）"""
    rows = []
    for level_str, res_dict in data.items():
        try:
            level = int(level_str)
        except ValueError:
            continue
        if level not in LEVELS:
            continue

        for res, model_dict in res_dict.items():
            if res not in RESOLUTIONS:
                print(res)
                continue
            # print(res)
            for model, drama_dict in model_dict.items():
                if model not in MODELS:
                    print(model)
                    continue
                # print(model)

                for drama, stats in drama_dict.items():
                    if drama not in DRAMA_NAMES:
                        if drama =='da_qin_di_guo_zhi_zong_heng':
                            drama='da_qin_di_guo'
                        else:
                            continue

                    # 主统计数据
                    ts = stats.get("total_stat", {})
                    # mini_batch统计数据
                    mb = ts.get("mini_batch", {})

                    # 构建一行数据（包含主数据和mini_batch）
                    rows.append({
                        # 维度信息
                        "level": level,
                        "resolution": res,
                        "model": model,
                        "drama": drama,
                        
                        # total_stat 主指标
                        "total": ts.get("total", 0),
                        "correct": ts.get("correct", 0),
                        "wrong": ts.get("wrong", 0),
                        "failed": ts.get("failed", 0),
                        "correct_weight": ts.get("correct_weight", 0.0),
                        "accuracy": ts.get("accuracy", 0.0),
                        "accuracy_weight": ts.get("accuracy_weight", 0.0),
                        
                        # mini_batch 子指标
                        "mini_batch_total": mb.get("total", 0),
                        "mini_batch_correct": mb.get("correct", 0),
                        "mini_batch_wrong": mb.get("wrong", 0),
                        "mini_batch_failed": mb.get("failed", 0),
                        "mini_batch_correct_weight": mb.get("correct_weight", 0.0),
                        "mini_batch_accuracy": mb.get("accuracy", 0.0),
                        "mini_batch_accuracy_weight": mb.get("accuracy_weight", 0.0),
                    })
    return rows

def aggregate_summary(df):
    """汇总数据（包含mini_batch）"""
    # 定义需要汇总的字段
    agg_columns = {
        # total_stat 汇总
        "total": "sum",
        "correct": "sum",
        "wrong": "sum",
        "failed": "sum",
        "correct_weight": "sum",
        # mini_batch 汇总
        "mini_batch_total": "sum",
        "mini_batch_correct": "sum",
        "mini_batch_wrong": "sum",
        "mini_batch_failed": "sum",
        "mini_batch_correct_weight": "sum",
    }

    # 按 level + resolution + model 汇总
    summary = df.groupby(["level", "resolution", "model"]).agg(agg_columns).reset_index()

    # 计算加权准确率（主指标）
    df["acc_total"] = df["accuracy"] * df["total"]
    acc_weight = df.groupby(["level", "resolution", "model"])["acc_total"].sum().reset_index(name="acc_total_sum")
    summary = summary.merge(acc_weight, on=["level", "resolution", "model"])
    summary["accuracy_weighted"] = round(summary["acc_total_sum"] / summary["total"], 4)

    # 计算mini_batch加权准确率（子指标）
    df["mini_batch_acc_total"] = df["mini_batch_accuracy"] * df["mini_batch_total"]
    mb_acc_weight = df.groupby(["level", "resolution", "model"])["mini_batch_acc_total"].sum().reset_index(name="mb_acc_total_sum")
    summary = summary.merge(mb_acc_weight, on=["level", "resolution", "model"])
    summary["mini_batch_accuracy_weighted"] = round(
        summary["mb_acc_total_sum"] / summary["mini_batch_total"].replace(0, 1), 4  # 避免除0
    )

    # 清理临时列
    summary = summary.drop(columns=["acc_total_sum", "mb_acc_total_sum"])
    
    return summary

def save_to_excel(drama_df, summary_df):
    """保存到Excel（包含明细和汇总）"""
    try:
        with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
            # 每部剧明细（包含mini_batch）
            drama_df.to_excel(writer, sheet_name="每部剧明细(含mini_batch)", index=False)
            # 维度汇总（包含mini_batch）
            summary_df.to_excel(writer, sheet_name="维度汇总(含mini_batch)", index=False)
        print(f"✅ Excel报告已保存：{EXCEL_FILE}")
    except Exception as e:
        print(f"❌ 保存Excel失败：{e}")

if __name__ == "__main__":
    # 1. 加载数据
    data = load_json(JSON_FILE)
    if not data:
        exit(1)
    
    # 2. 收集每部剧明细数据
    drama_rows = collect_all_drama_data(data)
    df_detail = pd.DataFrame(drama_rows)
    
    # 3. 生成汇总数据
    df_summary = aggregate_summary(df_detail)
    
    # 4. 打印基础信息
    print(f"\n📊 数据统计：")
    print(f"   - 每部剧明细条数：{len(df_detail)}")
    print(f"   - 维度汇总组合数：{len(df_summary)}")
    
    # 5. 保存到Excel
    save_to_excel(df_detail, df_summary)


📊 数据统计：
   - 每部剧明细条数：767
   - 维度汇总组合数：62
✅ Excel报告已保存：model_performance_report.xlsx


In [13]:
import json
import pandas as pd

# ====================== 固定配置 ======================
MODELS = [
    "qwen2.5-vl-72b",
    "qwen3-vl-32b",
    "qwen3.5-27b",
    "qwen3-vl-8b",
    "glm4.6v",
    "conan",
    "qwen3-vl-32b-thinking",
    "qwen3-vl-8b-thinking",
    "gemini-3.1-pro",
    "longvideoR1"
]

DRAMA_NAMES = [
    "kuang_biao", "huan_le_song", "zhan_chang_sha",
    "yi_qi_tong_guo_chuang_1", "ren_shi_jian", "friends",
    "da_qin_di_guo", "lost", "downton_abbey",
    "san_ti", "chen_mo_de_zhen_xiang", "shan_hai_qing", "zhen_huan_zhuan"
]

LEVELS = [3, 4, 5, 6]
RESOLUTIONS = ['256', '512', 'DA']  # 永远全部包含
JSON_FILE = "nested_result_without_qa_details.json"
EXCEL_FILE = "model_performance_report.xlsx"

# 维度大类-小类映射
B2S_dim = {
    "Visual Perception": [
        "Character’s Clothing", "Character Hairstyle", "Distinctive Characteristics of the Character",
        "Object Appearance", "Number of objects", "Whether there is", "Season, Weather Conditions",
        "Scene, Geographical Location", "Text in the picture",
        '人物穿着', '人物发型', '人物明显特征', '物体外观', '是否存在',
        '季节、天气情况', '场景、地理位置', '画面文字'
    ],
    'Spatial Understanding': [
        "Spatial location", "Comparison of Spatial Positions", "Shooting Perspective",
        '空间位置', '空间位置比较', '拍摄视角'
    ],
    'Action and Interaction': [
        "Character body movements", "Action Claim Subject", "Patient", "Object motion",
        '人物肢体动作', '动作主张主体', '动作承受者', '物体运动'
    ],
    'Event Understanding': [
        "Event Summary and Classification", "Event Background", "Event Outcome",
        "Character’s Purpose and Motivation",
        '事件概括、分类', '事件背景', '事件结局', '人物目的动机'
    ],
    'Temporal Understanding': [
        "Character Changes", "State Change", "Temporal sequence", "Temporal Comparison",
        "Object Attribute Transformation",
        '人物变化', '状态变化', '时序', '时序比较', '物体属性变换'
    ],
    'Relation & Logical Reasoning': [
        "The relationship between humans and objects", "Character Relationships",
        "Event Causal Logic", "Multi-step causal reasoning",
        '人与物体的关系', '人物关系', '事件因果逻辑', '多步原因推理'
    ],
    'Counting': [
        "Counting", "Quantity Comparison", "Number of objects",
        '计数', '数量比较', '物体数量'
    ]
}

SMALL_TO_BIG_DIM = {s: b for b, sl in B2S_dim.items() for s in sl}
# ======================================================

def load_json(path):
    """加载JSON文件（增加异常处理）"""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"❌ 加载JSON失败: {e}")
        return {}

def collect_all_data(data):
    main_rows = []
    dim_rows = []

    # 先打印JSON中的模型列表，验证longvideoR1是否存在
    all_models_in_json = set()
    for level_str, res_dict in data.items():
        for res, model_dict in res_dict.items():
            all_models_in_json.update(model_dict.keys())
    print(f"📌 JSON中包含的所有模型: {sorted(all_models_in_json)}")
    print(f"📌 是否包含longvideoR1: {'longvideoR1' in all_models_in_json}")

    for level_str, res_dict in data.items():
        try:
            level = int(level_str)
        except:
            continue
        if level not in LEVELS:
            continue

        for res, model_dict in res_dict.items():
            if res not in RESOLUTIONS:
                continue

            # 遍历所有模型，重点处理longvideoR1的特殊字段
            for model, drama_dict in model_dict.items():
                # 严格匹配模型名称（包括longvideoR1）
                if model not in MODELS:
                    continue
                
                # 打印当前处理的模型，验证longvideoR1是否被命中
                if model == "longvideoR1":
                    print(f"✅ 正在处理longvideoR1数据 (分辨率: {res}, level: {level})")

                for drama, stats in drama_dict.items():
                    if drama not in DRAMA_NAMES:
                        if drama == 'da_qin_di_guo_zhi_zong_heng':
                            drama = 'da_qin_di_guo'
                        else:
                            continue

                    # 1. 收集主数据（所有模型通用）
                    ts = stats.get("total_stat", {})
                    mb = ts.get("mini_batch", {})
                    main_rows.append({
                        "level": level,
                        "resolution": res,
                        "model": model,
                        "drama": drama,
                        "total": ts.get("total", 0),
                        "correct": ts.get("correct", 0),
                        "wrong": ts.get("wrong", 0),
                        "failed": ts.get("failed", 0),
                        "correct_weight": ts.get("correct_weight", 0.0),
                        "accuracy": ts.get("accuracy", 0.0),
                        "accuracy_weight": ts.get("accuracy_weight", 0.0),
                        "mini_total": mb.get("total", 0),
                        "mini_correct": mb.get("correct", 0),
                        "mini_wrong": mb.get("wrong", 0),
                        "mini_failed": mb.get("failed", 0),
                        "mini_correct_weight": mb.get("correct_weight", 0.0),
                        "mini_accuracy": mb.get("accuracy", 0.0),
                        "mini_accuracy_weight": mb.get("accuracy_weight", 0.0),
                    })

                    # 2. 收集维度数据（适配longvideoR1的特殊字段）
                    # 优先读取longvideoR1的sub_dimension_stat，其他模型读取dimension_stat
                    if model == "longvideoR1":
                        dim_stat = stats.get("sub_dimension_stat", {})
                        if dim_stat:
                            print(f"✅ 读取到longvideoR1的sub_dimension_stat数据 (drama: {drama}, 维度数: {len(dim_stat)})")
                        else:
                            print(f"⚠️ longvideoR1无sub_dimension_stat数据 (drama: {drama})")
                    else:
                        dim_stat = stats.get("dimension_stat", {})

                    # 处理维度数据（通用逻辑）
                    for small_dim, dstat in dim_stat.items():
                        if small_dim not in SMALL_TO_BIG_DIM:
                            continue
                        big_dim = SMALL_TO_BIG_DIM[small_dim]
                        dmini = dstat.get("mini_batch", {})

                        dim_rows.append({
                            "level": level,
                            "resolution": res,  # 保留DA分辨率
                            "model": model,     # 保留longvideoR1
                            "drama": drama,
                            "small_dim": small_dim,
                            "big_dim": big_dim,
                            "dim_total": dstat.get("total", 0),
                            "dim_correct": dstat.get("correct", 0),
                            "dim_wrong": dstat.get("wrong", 0),
                            "dim_failed": dstat.get("failed", 0),
                            "dim_correct_weight": dstat.get("correct_weight", 0.0),
                            "dim_accuracy": dstat.get("accuracy", 0.0),
                            "dim_accuracy_weight": dstat.get("accuracy_weight", 0.0),
                            "dim_mini_total": dmini.get("total", 0),
                            "dim_mini_correct": dmini.get("correct", 0),
                            "dim_mini_wrong": dmini.get("wrong", 0),
                            "dim_mini_failed": dmini.get("failed", 0),
                            "dim_mini_correct_weight": dmini.get("correct_weight", 0.0),
                            "dim_mini_accuracy": dmini.get("accuracy", 0.0),
                            "dim_mini_accuracy_weight": dmini.get("accuracy_weight", 0.0),
                        })

    # 验证收集结果
    df_dim_temp = pd.DataFrame(dim_rows)
    model_in_dim = df_dim_temp['model'].unique()
    print(f"\n📌 维度数据中包含的模型: {sorted(model_in_dim)}")
    print(f"📌 longvideoR1维度数据行数: {len(df_dim_temp[df_dim_temp['model'] == 'longvideoR1'])}")
    if len(df_dim_temp[df_dim_temp['model'] == 'longvideoR1']) > 0:
        print(f"📌 longvideoR1维度数据分辨率分布: \n{df_dim_temp[df_dim_temp['model'] == 'longvideoR1']['resolution'].value_counts()}")

    return main_rows, dim_rows

def agg_main(df):
    agg = {
        "total": "sum",
        "correct": "sum",
        "wrong": "sum",
        "failed": "sum",
        "correct_weight": "sum",
        "accuracy": "mean",
        "accuracy_weight": "mean",
        "mini_total": "sum",
        "mini_correct": "sum",
        "mini_wrong": "sum",
        "mini_failed": "sum",
        "mini_correct_weight": "sum",
        "mini_accuracy": "mean",
        "mini_accuracy_weight": "mean",
    }
    summary = df.groupby(["level", "resolution", "model"], dropna=False).agg(agg).reset_index()
    summary["accuracy_total_weighted"] = (summary["correct_weight"] / summary["total"].replace(0, 1)).round(4)
    summary["mini_accuracy_total_weighted"] = (summary["mini_correct_weight"] / summary["mini_total"].replace(0, 1)).round(4)
    return summary

def agg_dimension(df_dim):
    # 小维度汇总
    agg_small = {
        "dim_total": "sum",
        "dim_correct": "sum",
        "dim_wrong": "sum",
        "dim_failed": "sum",
        "dim_correct_weight": "sum",
        "dim_accuracy": "mean",
        "dim_accuracy_weight": "mean",
        "dim_mini_total": "sum",
        "dim_mini_correct": "sum",
        "dim_mini_wrong": "sum",
        "dim_mini_failed": "sum",
        "dim_mini_correct_weight": "sum",
        "dim_mini_accuracy": "mean",
        "dim_mini_accuracy_weight": "mean",
    }
    small_summary = df_dim.groupby(["level", "resolution", "model", "small_dim", "big_dim"], dropna=False).agg(agg_small).reset_index()

    # 大类汇总
    agg_big = {
        "dim_total": "sum",
        "dim_correct": "sum",
        "dim_wrong": "sum",
        "dim_failed": "sum",
        "dim_correct_weight": "sum",
        "dim_accuracy": "mean",
        "dim_accuracy_weight": "mean",
        "dim_mini_total": "sum",
        "dim_mini_correct": "sum",
        "dim_mini_wrong": "sum",
        "dim_mini_failed": "sum",
        "dim_mini_correct_weight": "sum",
        "dim_mini_accuracy": "mean",
        "dim_mini_accuracy_weight": "mean",
    }
    big_summary = small_summary.groupby(["level", "resolution", "model", "big_dim"], dropna=False).agg(agg_big).reset_index()

    big_summary.rename(columns={
        "dim_accuracy": "big_dim_accuracy",
        "dim_accuracy_weight": "big_dim_accuracy_weight",
        "dim_mini_accuracy": "big_dim_mini_accuracy",
        "dim_mini_accuracy_weight": "big_dim_mini_accuracy_weight",
    }, inplace=True)

    # 验证大类汇总中是否包含longvideoR1
    print(f"\n📌 大类维度汇总中包含的模型: {sorted(big_summary['model'].unique())}")

    return small_summary, big_summary

def save_all(df_main, df_main_agg, df_dim_detail, df_small_agg, df_big_agg):
    with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as w:
        df_main.to_excel(w, sheet_name="01_每部剧明细", index=False)
        df_main_agg.to_excel(w, sheet_name="02_主数据汇总", index=False)
        df_dim_detail.to_excel(w, sheet_name="03_维度明细(小类)", index=False)
        df_small_agg.to_excel(w, sheet_name="04_小维度汇总", index=False)
        df_big_agg.to_excel(w, sheet_name="05_大类维度汇总", index=False)
    
    # 最终验证
    print(f"\n✅ 导出完成！")
    print(f"   - 03_维度明细(小类) 包含longvideoR1: {'longvideoR1' in df_dim_detail['model'].values}")
    print(f"   - 04_小维度汇总 包含longvideoR1: {'longvideoR1' in df_small_agg['model'].values}")
    print(f"   - 05_大类维度汇总 包含longvideoR1: {'longvideoR1' in df_big_agg['model'].values}")

if __name__ == "__main__":
    data = load_json(JSON_FILE)
    main_rows, dim_rows = collect_all_data(data)

    df_main = pd.DataFrame(main_rows)
    df_dim = pd.DataFrame(dim_rows)

    # 额外验证：longvideoR1主数据和维度数据的DA分辨率
    longvideo_main_da = df_main[(df_main['model'] == 'longvideoR1') & (df_main['resolution'] == 'DA')]
    longvideo_dim_da = df_dim[(df_dim['model'] == 'longvideoR1') & (df_dim['resolution'] == 'DA')]
    print(f"\n📌 longvideoR1 DA分辨率主数据行数: {len(longvideo_main_da)}")
    print(f"📌 longvideoR1 DA分辨率维度数据行数: {len(longvideo_dim_da)}")

    df_main_agg = agg_main(df_main)
    df_small_agg, df_big_agg = agg_dimension(df_dim)

    save_all(df_main, df_main_agg, df_dim, df_small_agg, df_big_agg)

📌 JSON中包含的所有模型: ['conan', 'gemini-3.1-pro', 'glm4.6v', 'qwen2.5-vl-72b', 'qwen3-vl-32b', 'qwen3-vl-32b-thinking', 'qwen3-vl-8b', 'qwen3-vl-8b-thinking', 'qwen3.5-27b']
📌 是否包含longvideoR1: False

📌 维度数据中包含的模型: ['conan', 'gemini-3.1-pro', 'glm4.6v', 'qwen2.5-vl-72b', 'qwen3-vl-32b', 'qwen3-vl-32b-thinking', 'qwen3-vl-8b', 'qwen3-vl-8b-thinking', 'qwen3.5-27b']
📌 longvideoR1维度数据行数: 0

📌 longvideoR1 DA分辨率主数据行数: 0
📌 longvideoR1 DA分辨率维度数据行数: 0

📌 大类维度汇总中包含的模型: ['conan', 'gemini-3.1-pro', 'glm4.6v', 'qwen2.5-vl-72b', 'qwen3-vl-32b', 'qwen3-vl-32b-thinking', 'qwen3-vl-8b', 'qwen3-vl-8b-thinking', 'qwen3.5-27b']

✅ 导出完成！
   - 03_维度明细(小类) 包含longvideoR1: False
   - 04_小维度汇总 包含longvideoR1: False
   - 05_大类维度汇总 包含longvideoR1: False
